# Groundwater transport (GWT) model

This notebook builds and runs a groundwater transport (GWT) simulation for the synthetic valley. It reuses the heads and flows from the groundwater flow (GWF) model that is built and run in [`flopy-intro-gwt-a`](flopy-intro-gwt-a.ipynb) (notebook A), so **run that notebook first**.

> **Model reference.** The synthetic valley model used in these notebooks is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

## What this notebook covers

Add solute transport on top of the flow model from
[`flopy-intro-gwt-a`](flopy-intro-gwt-a.ipynb), coupled through the saved flow
output rather than solved with it.

By the end of this notebook you will be able to:

- build a groundwater transport (**GWT**) model on a grid taken from a flow
  model,
- connect it to saved heads and flows with the Flow Model Interface (**FMI**),
- name a boundary as a solute source with the Source and Sink Mixing (**SSM**)
  package, and
- plot concentrations through time.


## How transport builds on flow

This notebook adds **solute transport** on top of the flow model from notebook A.

**Two models, one direction of coupling**

- the **groundwater-flow model** (`GWF`, from notebook A) computes where the water is and how fast it moves — heads and cell-by-cell flows;
- the **groundwater-transport model** (`GWT`, built here) computes where a dissolved substance goes — concentrations — by being carried along by that flow field.

The coupling is **one-way**: GWT reads the heads and flows that GWF already saved to disk and does not change them. That is why notebook A must be run first, and why this notebook starts by checking that the flow output exists.

**The GWT packages**

The transport model uses the **same grid** as the flow model and is assembled from packages, each contributing one piece of the transport equation:

- `FMI` (Flow Model Interface) — reads the saved GWF heads and flows; this is the link between the two models;
- `IC` — the initial concentration everywhere (zero here);
- `ADV` (advection) — solute carried along with the moving water;
- `DSP` (dispersion) — spreading and mixing as the solute moves;
- `MST` (mobile storage and transfer) — how much solute the pore space can hold (uses the porosity). Here the porosity is the flow model's specific yield, the share of the aquifer that drains by gravity and so the connected pore space the water moves through: 0.1 in layer 1, 0.25 in the rest of the aquifer, and 0.45 in the clay. With a porosity of 0.1, the same flow carries solute 2.5 times as fast through layer 1 as through layers 2, 4, and 5;
- `SSM` (source-sink mixing) — sets the concentration of the water that boundaries add to the aquifer, here from the **lake** (`LAK`);
- `OC` — controls which concentrations are saved.

**Where the solute comes from**

Notebook A gave the lake an auxiliary variable, `CONCENTRATION`, of 100, and the flow model wrote it to the budget file with every flow between the lake and the aquifer. SSM reads it, so lake water leaking into the aquifer carries a concentration of 100, and a simulated concentration reads as the percent of the water in a cell that came from the lake. Water from the boundaries SSM does not name — rain soaking in from the unsaturated zone, the stream — enters with a concentration of zero. Water leaving the aquifer, to the wells, the drains, the stream, or evapotranspiration, leaves at the concentration of the cell it leaves from. The transport model does not follow the solute inside the lake, the stream, or the unsaturated zone; that takes the advanced transport packages. It only needs the concentration of the water as it crosses into the aquifer.

**The lifecycle is the same as flow**

Just like the flow model, you build the simulation and its solver, add the model and its packages, then `write_simulation()` and `run_simulation()`. The result is a concentration field saved through time, which the figures map in plan view and cross section.

Keep this picture — *flow first, then transport reads it; same grid; packages assemble the transport equation* — in mind as you go.

## Imports and helper functions

Import FloPy and the plotting libraries used throughout the notebook. The `time_slider_view` helper (from `mf6_notebook_helpers`) draws a single interactive figure that updates as you move a time slider; it renders each frame to a PNG so exactly one figure is shown in both VS Code and JupyterLab.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from mf6_notebook_helpers import (
    display_ticks,
    label_map_axes,
    length_name,
    require_gwf_output,
    time_slider_view,
)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

## Load the existing flow model

Load the flow model from the `models/` workspace where notebook A wrote and ran it. Read a few quantities from it (layer and cell counts, the number of stress periods, and the center of the lake) that the transport model and figures need.

In [ ]:
# the gwf model was built and run in flopy-intro-gwt-a.ipynb; load that
# already-run model from its models/ workspace

name = "sv"
new_ws = pl.Path("models/synthetic-valley-vg-advanced-annual")
gwt_ws = new_ws.parent / "synthetic-valley-vg-advanced-annual-gwt"

In [ ]:
# exercise: load the flow simulation that flopy-intro-gwt-a built and ran
sim = flopy.mf6.MFSimulation.load(
    sim_name=name,
    sim_ws=new_ws,
    write_headers=False,
)

In [ ]:
# exercise: get the flow model, and read its grid size and stress periods from it
gwf = sim.get_model(name)
nlay, ncpl = gwf.disv.nlay.array, gwf.disv.ncpl.array
shape3d = (nlay, ncpl)
# end exercise

nper = sim.tdis.nper.array
perlen = sim.tdis.perioddata.array["perlen"]

# x-coordinate of the center of the lake (LAK) package; the model cross
# sections (A-A') are taken along this north-south line so they cut through
# the middle of the lake instead of the center of the model domain
lak_cellids = gwf.lak.connectiondata.array["cellid"]
lak_icpl = np.array(sorted({c[-1] for c in lak_cellids}))
lake_center_x = float(gwf.modelgrid.xcellcenters[lak_icpl].mean())

# map location of the two pumping wells and the layers their screens are in
maw_cellids = gwf.maw.connectiondata.array["cellid"]
maw_icpl = sorted({c[-1] for c in maw_cellids})
maw_layers = sorted({c[0] for c in maw_cellids})
well_x = gwf.modelgrid.xcellcenters[maw_icpl]
well_y = gwf.modelgrid.ycellcenters[maw_icpl]

## Check the flow model has been run

The transport model reads heads, cell-by-cell flows, and cell saturation directly from the flow model, so that output must already exist. This check raises a clear error if the head or budget file is missing, empty, or contains no time steps, pointing you back to notebook A.

In [ ]:
# the transport model reads heads and flows from the gwf model, so that output
# must already exist. require_gwf_output (in mf6_notebook_helpers) fails early with a
# clear message pointing back to notebook A if the head or budget file is missing.
require_gwf_output(gwf, new_ws)

## Build the transport model

Set up the GWT model on the same grid as the flow model. The flow-model interface (FMI) package feeds it the saved heads and flows, the source-sink mixing (SSM) package reads the lake's concentration, and the advection, dispersion, and storage packages control how the solute spreads. As in notebook A, all inputs are in meters and days (so the dispersivities `alh` and `ath1` are in meters).

### Create the simulation and solver

Create a new simulation for the transport model, copy the flow model's time discretization, and configure the iterative solver. The transport model takes the same steps as the flow model: the 16,071-day steady-state period in one step, then 20 one-year steps.

In [ ]:
gwt_name = f"{name}_gwt"

In [ ]:
# exercise: create the transport simulation, its TDIS (reusing the flow model's periods), and its solver
sim_gwt = flopy.mf6.MFSimulation(
    sim_name=gwt_name,
    sim_ws=gwt_ws,
)

perioddata = sim.tdis.perioddata.array
tdis = flopy.mf6.ModflowTdis(
    sim_gwt,
    time_units="DAYS",
    nper=nper,
    perioddata=perioddata,
)
ims = flopy.mf6.ModflowIms(
    sim_gwt,
    filename=f"{gwt_name}.ims",
    print_option="all",
    complexity="simple",
    outer_maximum=500,
    inner_maximum=200,
    linear_acceleration="bicgstab",
)

### Add the transport model and packages

Add the GWT model and its packages: grid (**DISV**, discretization by vertices), initial concentration, advection, dispersion, mobile storage, the flow interface, the lake solute source, and output control.

In [ ]:
# exercise: create the transport model and give it the same DISV grid as the flow model
gwt = flopy.mf6.ModflowGwt(
    sim_gwt,
    modelname=gwt_name,
)
dis = flopy.mf6.ModflowGwtdisv(
    gwt,
    length_units="meters",
    nlay=nlay,
    ncpl=ncpl,
    cell2d=gwf.disv.cell2d.array,
    nvert=gwf.disv.nvert.array,
    vertices=gwf.disv.vertices.array,
    top=gwf.disv.top.array,
    botm=gwf.disv.botm.array,
    idomain=gwf.disv.idomain.array,
)
# end exercise
# exercise: add initial conditions (zero concentration), advection (utvd), and dispersion
ic = flopy.mf6.ModflowGwtic(
    gwt,
    strt=0.0,
)
adv = flopy.mf6.ModflowGwtadv(
    gwt,
    scheme="utvd",
)
dsp = flopy.mf6.ModflowGwtdsp(
    gwt,
    diffc=0.0e-12,
    alh=75.0,
    ath1=7.5,
)
# end exercise
# exercise: add mass storage, using the flow model's specific yield as the porosity
mst = flopy.mf6.ModflowGwtmst(
    gwt,
    porosity=gwf.sto.sy.array,
)
# end exercise
# exercise: point the Flow Model Interface at the head and budget files the flow model wrote
pd = [
    ("GWFHEAD", f"../{new_ws.name}/{name}.hds", None),
    ("GWFBUDGET", f"../{new_ws.name}/{name}.cbc", None),
]
fmi = flopy.mf6.ModflowGwtfmi(
    gwt,
    packagedata=pd,
)
# end exercise
# lake water enters at the lake's CONCENTRATION auxiliary variable; water from
# every other boundary enters at zero
sourcerecarray = [
    ("LAK-1", "AUX", "CONCENTRATION"),
]
ssm = flopy.mf6.ModflowGwtssm(
    gwt,
    sources=sourcerecarray,
)
oc = flopy.mf6.ModflowGwtoc(
    gwt,
    concentration_filerecord=f"{name}.ucn",
    saverecord={
        0: [
            ("CONCENTRATION", "LAST"),
        ]
    },
    printrecord=[("BUDGET", "ALL")],
)

## Run the transport model

Write the transport input files and run the simulation.

In [ ]:
# exercise: write the transport input files
sim_gwt.write_simulation()

In [ ]:
# exercise: run the transport model
success, buff = sim_gwt.run_simulation()
assert success, "MODFLOW 6 did not terminate normally"

## View the simulated concentrations

Plot the simulated solute concentrations. Use the time slider to step through the simulation.

### Inspect the concentration output

A quick look at the concentration output: the range of simulated values and the data at the first output time.

In [ ]:
# exercise: read the output times and the final concentration, and report its range
times = gwt.output.concentration().get_times()
conc = gwt.output.concentration().get_data(totim=times[-1])
conc[conc < 1e30].min(), conc[conc < 1e30].max()  # end exercise

### Plot settings

Shared figure size and contour styling used by the concentration figures below.

In [ ]:
six_panel_figsize = (17.15 / 2.541, 1.4 * 0.8333 * 17.15 / 2.541)
mosaic = [[0, 1, 2], [3, 4, "."]]

In [ ]:
contour_color = "white"
contour_style = "--"

sv_gwt_contour_dict = {
    "linewidths": 0.75,
    "colors": contour_color,
    "linestyles": contour_style,
}

### Concentration maps through time

Simulated concentration in each layer at the selected time, on a log color scale. The stream is drawn in cyan and the two pumping wells, screened in layers 4 and 5, in red.

In [ ]:
times = gwt.output.concentration().get_times()

extent = gwt.modelgrid.extent
xc = lake_center_x  # cross-section through the center of the lake (A-A')

# reuse the six-panel layout but turn the empty slot into a colorbar axes
conc_mosaic = [[0, 1, 2], [3, 4, "cbar"]]


def plot_concentration(time_index):
    totim = times[time_index]
    conc = gwt.output.concentration().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            conc_mosaic,
            figsize=six_panel_figsize,
            layout="constrained",
        )
        fig.suptitle(f"Time = {totim:g} days (time index {time_index})")

        norm = mpl.colors.LogNorm(vmin=0.01, vmax=100)
        for k in range(nlay):
            ax = axd[k]
            ax.set_xlim(extent[:2])
            ax.set_ylim(extent[2:])
            ax.set_title(f"Layer {k + 1}")
            mm = flopy.plot.PlotMapView(
                model=gwt,
                ax=ax,
                extent=gwt.modelgrid.extent,
                layer=k,
            )
            mm.plot_ibound(color_vpt="blue", alpha=0.25)
            pa = mm.plot_array(
                conc,
                edgecolor="black",
                lw=0.25,
                masked_values=[0],
                norm=norm,
            )

            mm.plot_bc(package=gwf.sfr, color="cyan")
            if k in maw_layers:
                ax.plot(well_x, well_y, "o", mfc="red", mec="black", ms=4)
            label_map_axes(ax, xlabel=False, ylabel=False)
            display_ticks(ax.xaxis, nbins=3)

            c = conc[k]
            cmax = c[c < 1e30].max()
            contour_array = cmax >= 0.1
            if contour_array:
                cs = mm.contour_array(
                    conc,
                    # masked_values=[0],
                    levels=[0.01, 1, 10, 100],
                    **sv_gwt_contour_dict,
                )

                ax.clabel(
                    cs,
                    inline=True,
                    fmt="%g",
                    fontsize=8,
                    inline_spacing=0.5,
                    colors="white",
                )

            # show the cross-section location on the Layer 1 panel
            if k == 0:
                ax.plot([xc, xc], extent[2:], color="red", lw=1.5, ls="--")
                ax.text(
                    xc,
                    extent[3],
                    "A",
                    color="red",
                    fontweight="bold",
                    ha="center",
                    va="bottom",
                )
                ax.text(
                    xc,
                    extent[2],
                    "A'",
                    color="red",
                    fontweight="bold",
                    ha="center",
                    va="top",
                )

        # slim vertical colorbar centered in the empty mosaic panel
        axd["cbar"].axis("off")
        cax = inset_axes(
            axd["cbar"],
            width="25%",
            height="85%",
            loc="center",
        )
        ticks = [0.01, 0.1, 1, 10, 100]
        cb = fig.colorbar(pa, cax=cax, ticks=ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.minorticks_off()
        cb.set_label("Concentration (percent lake water)")

    return fig


time_slider_view(plot_concentration, len(times))

**What to look for.** Concentration is drawn on a **log color scale**, so each step in color is a factor of ten, and a concentration reads as the percent of the water in a cell that came from the lake. Lake water leaks down beneath the lake and moves south with the groundwater, past the ridge and down the western side of the valley. By the end of the 16,071-day steady-state period (index 0) it makes up 1 percent or more of the water in 16,537 of the 26,801 active cells, reaching to within 380 m (1,230 ft) of the southern edge but no farther east than 1,960 m (6,410 ft); the eastern half of the valley gets its water from recharge. Through the 20 pumping years the lake level falls and the lake leaks less, so by the last of the 21 output times the cells at 1 percent or more drop to 14,683. At the southwestern well, which pumps from layers 4 and 5, lake water rises from 5.3 to 6.2 percent when pumping starts to 10.2 to 10.7 percent at the end; at the northeastern well it stays below 0.01 percent.

### Concentration along section A–A' through time

Simulated concentration in cross section along A–A', shown only in the saturated part of each layer.

In [ ]:
# simulated concentration along section A-A' (center of the lake) through time
# (same settings as the concentration map slider figure above)
extent = gwt.modelgrid.extent
xc = lake_center_x  # cross-section through the center of the lake (A-A')
line = {"line": [(xc, extent[3]), (xc, extent[2])]}  # A (top) -> A' (bottom)

times = gwt.output.concentration().get_times()
conc_levels = [0.01, 1, 10, 100]
conc_ticks = [0.01, 0.1, 1, 10, 100]
xs_w = 17.15 / 2.541


def plot_concentration_xs(time_index):
    totim = times[time_index]
    conc = gwt.output.concentration().get_data(totim=totim)
    # head at the same time so concentrations are only shown in the
    # saturated portion of each layer
    head = gwf.output.head().get_data(totim=totim)

    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(xs_w, 0.5 * xs_w), layout="constrained")
        ax.set_title(
            f"Concentration along A-A', time = {totim:g} days (index {time_index})"
        )
        xs = flopy.plot.PlotCrossSection(
            model=gwt,
            ax=ax,
            line=line,
        )

        norm = mpl.colors.LogNorm(vmin=0.01, vmax=100)
        pa = xs.plot_array(conc, head=head, masked_values=[0], norm=norm)
        xs.plot_grid(lw=0.3, color="0.5")
        xs.plot_ibound()

        cmax = conc[conc < 1e30].max()
        if cmax >= conc_levels[0]:
            cs = xs.contour_array(
                conc,
                head=head,
                levels=conc_levels,
                **sv_gwt_contour_dict,
            )
            ax.clabel(
                cs,
                inline=True,
                fmt="%g",
                fontsize=8,
                inline_spacing=0.5,
                colors=contour_color,
            )

        label_map_axes(ax, xlabel=False, ylabel=False)
        ax.set_xlabel(f"Distance along section, in {length_name()}")
        ax.set_ylabel(f"Elevation, in {length_name()}")

        # label the section endpoints to match the map figure
        ax.text(
            0.0,
            1.01,
            "A",
            color="red",
            fontweight="bold",
            ha="left",
            va="bottom",
            transform=ax.transAxes,
        )
        ax.text(
            1.0,
            1.01,
            "A'",
            color="red",
            fontweight="bold",
            ha="right",
            va="bottom",
            transform=ax.transAxes,
        )

        cb = fig.colorbar(pa, ax=ax, shrink=0.8, ticks=conc_ticks)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
        cb.minorticks_off()
        cb.set_label("Concentration (percent lake water)")

    return fig


time_slider_view(plot_concentration_xs, len(times))

**What to look for.** The cross section shows concentration only within the saturated part of each layer (it uses the head field for that), so the colored area is the wetted aquifer. Lake water fills the whole thickness of the aquifer beneath the lake, clay included, and is strongest, up to 81 percent, at the south shore of the lake. South of the ridge the share of lake water falls off toward the southern edge, to below 1 percent near the end of the section.

**Recap.** Building on the flow solution from notebook A, you created a transport model on the same grid, connected it to the saved heads and flows through the FMI package, labeled the water leaking from the lake through SSM, and ran it to produce concentrations through time. This flow-then-transport workflow — run GWF, then let GWT use its results — is the standard way MODFLOW 6 simulates the movement of dissolved substances.